## Historical NDVI processing

Here we applied the method formulated in the previous report to all the pixel at once.
Due to the huge size of the data, we use xarray and Dask to parallelize the computation and efficently use the CPU resource.

### Step 1: dataset creation

After downloading the dataset from Zenodo, we create another dataset that will be used for the analysis
This dataset contains:

- evenly spaced data at daily resolution, where no data is avaible we insert a placeholder
- the mean between the lower and upper double logistic bands, the array have the same dimension of ndvi

### Step 2: processing

After the dataset creation, we perform the following analysis on the full time series pixel-wise

- Run the outlier detection
- Run the estimation from the last observation to the last date, where the two do not coincide
- Run the linear deltas interpolatin from the last fourth date to the last date
- Run the smoothing from the first date to the last fourth date 
- Add the smoothed, interpolated and estimated deltas to the mean of the bands to have the final processed NDVI

In some cases no observation are avaible for the full time serie, we skip those pixels with no enough observation (7).

The analysis take 20 minutes for 1 milion pixels, since the total amount of pixels are 105 milions in total shoud take 26 hours. (not tested yet)

## Continous NDVI processing

Using the same model approach, we set up a workflow for the continous NDVI ingestion. It is composed of 2 steps similar as above.

### Step 1: Dataset creation

Identical to step 1 of the historical NDVI processing. In this case we add a boolean mask to idenitfy the last 7 observation. 

The boolean mask will have a **true** value when a date in found in the original dataset, otherwise **false**.

These 7 last observation will be used to perform the outlier detection, smoothing and gapfilling as discussed in the previous report.

### Step 2: processing

Similar to the historical NDVI processing, we firstly need to perform the outlier detection. 

### Step 2.1: outlier detection

- We use the boolean mask to identify the original NDVI values, from them we select only the ones in the meaningful range (0-1) to discard the placeholders
- once we have the observed NDVI, we calcuate the difference from the mean of the bands (called delta) and the difference from the delta neighobour (called delta-delta). If both of the values are above the threshold (0.1) the observation is flagged as outlier and removed.
- the last date does not have two neighobour. For this reason we evaluated it separately. If the delta and the single delta-delta are above the threshold, the observation is flagged as potential outlier.

After the outlier detection, if a timeserie does not have enough observation is ignored as before.

After the outleir detection, we select the last 7 observation and we perform the continous NDVI processing following this logic:

- Firstly, we check if the NDVI of the current date (hereinafter called current NDVI) is in the meanigful range or not.
- - if is not in the meanigful range it means that we do not have an observed NDVI, so we estimate it using the last known NDVI (hereinafter called last NDVI) and the current means of the bands.

- If the current NDVI is in the meanigful range, we already evaluated it in the step 2.1
- - If the current NDVI is flagged as potential outlier, we keep the original value. The potential outlier will be evaluated when a new observation will be avaible in step 2.1

- If the current NDVI is not flagged as potential outlier, it is considered a true observation and we perform the following action
- - linear deltas gapfilling from the last NDVI to the current NDVI
- - Smoothing and linear deltas interpolation from the third to the fourth of the last 7 observation

All the deltas above will be summed to the corresponded mean of the bands.

## Implementation on SATROMO

TODO